# String Formatting

**Session 05 (2026-10-17) · Assignment 2 of 3 (optional)**

**String formatting** means building a string from fixed text plus values, like `"Asha scored 92.5%"`. Python has three ways to do it. This notebook focuses on `str.format()`, which the assignment asks for, and compares it with f-strings and `%` formatting.

What you'll learn:

1. Why we need string formatting at all
2. `str.format()`: positional, indexed and named placeholders, and reusable templates
3. f-strings, the modern shortcut
4. `%` formatting, the old style
5. The format specification mini-language: decimals, width, alignment, signs, separators, number bases
6. Formatting dates
7. Dynamic widths and literal braces
8. Common mistakes
9. Which style to use when
10. Practice exercises, with solutions

> Run the cells from top to bottom. Examples that fail on purpose are wrapped in `try` / `except`, so **Run All** works.

---
## 1. Why do we need string formatting?

Joining strings with `+` gets messy quickly: every non-string value has to be converted with `str()`, and you can't control decimals or alignment.

In [1]:
name = "Asha"
score = 92.456

# + only joins strings, so adding a number raises an error
try:
    message = "Score: " + score
except TypeError as error:
    print("TypeError:", error)

# Works, but it's hard to read and shows every decimal
print(name + " scored " + str(score) + " out of 100")

# With formatting: readable, and the decimals are under control
print("{} scored {:.1f} out of 100".format(name, score))

TypeError: can only concatenate str (not "float") to str
Asha scored 92.456 out of 100
Asha scored 92.5 out of 100


---
## 2. `str.format()`

**What it is:** a string method. You write a **template** containing `{}` **placeholders**, then call `.format()` with the values that go into them.

**Syntax:** `"template with {} placeholders".format(value1, value2, ...)`

**Returns:** a **new** string. The template itself doesn't change, because strings are immutable.

**Behind the scenes:** for each placeholder, Python picks the matching argument, turns it into text with the built-in `format(value, spec)` function, and joins all the pieces together.

### 2.1 Positional placeholders

Empty `{}` are filled from left to right.

In [2]:
print("{} scored {}".format(name, score))
print("{} + {} = {}".format(2, 3, 2 + 3))  # arguments can be any expression

Asha scored 92.456
2 + 3 = 5


### 2.2 Indexed placeholders

`{0}`, `{1}`, ... pick an argument by its position, so you can reorder or repeat values.

In [3]:
print("{1} was scored by {0}".format(name, score))  # reorder
print("{0}, {0}, {0}! Go {0}!".format("Asha"))       # reuse one value

92.456 was scored by Asha
Asha, Asha, Asha! Go Asha!


### 2.3 Named placeholders

`{student}` is filled from a keyword argument with that name. Long templates become much easier to read.

In [4]:
print("{student} scored {marks} in {subject}".format(student="Ravi", marks=88, subject="Python"))

Ravi scored 88 in Python


### 2.4 Reusable templates and dictionaries

A big advantage of `.format()` over f-strings: the template can be written once, stored in a variable (or loaded from a file), and filled in later with different data. `**` unpacks a dictionary into named arguments.

In [5]:
# One template, reused for every student
template = "Dear {name}, your score is {score}/100. Result: {result}."

students = [
    {"name": "Asha", "score": 92, "result": "Pass"},
    {"name": "Ravi", "score": 45, "result": "Fail"},
]

for student in students:
    print(template.format(**student))  # ** turns the dict into name=..., score=..., result=...

Dear Asha, your score is 92/100. Result: Pass.
Dear Ravi, your score is 45/100. Result: Fail.


### 2.5 Reaching inside an argument

`{0[1]}` takes item 1 of a list argument, and `{0[city]}` takes a key from a dict argument. Note there are no quotes around the key inside the placeholder.

In [6]:
marks = [92, 88, 75]
person = {"name": "Monal", "city": "Pune"}

print("First mark: {0[0]}, last mark: {0[2]}".format(marks))
print("{0[name]} lives in {0[city]}".format(person))
print("{p[name]} lives in {p[city]}".format(p=person))  # works with named placeholders too

First mark: 92, last mark: 75
Monal lives in Pune
Monal lives in Pune


---
## 3. f-strings (formatted string literals)

**What it is:** put `f` before the opening quote, then write variables or **any expression** directly inside the `{}`. f-strings arrived in Python 3.6 and are the most common style in new code.

**Syntax:** `f"text {expression} text"`

**Returns:** a new string, built at the moment that line runs.

In [7]:
name, score = "Asha", 92.456

print(f"{name} scored {score}")
print(f"Next target: {score + 5:.1f}")               # any expression, then a format spec after the :
print(f"Name in capitals: {name.upper()}")           # method calls work too
print(f"Passed: {'yes' if score >= 50 else 'no'}")   # even a conditional expression

Asha scored 92.456
Next target: 97.5
Name in capitals: ASHA
Passed: yes


**Debugging shortcut:** `{variable=}` prints both the expression and its value (Python 3.8+).

In [8]:
print(f"{name=}, {score=}")
print(f"{score * 2 = }")  # spaces around the = are kept in the output

name='Asha', score=92.456
score * 2 = 184.912


The same output, written in all three styles:

In [9]:
print("{} scored {:.2f}".format(name, score))  # str.format()
print(f"{name} scored {score:.2f}")             # f-string
print("%s scored %.2f" % (name, score))         # % formatting

Asha scored 92.46
Asha scored 92.46
Asha scored 92.46


---
## 4. `%` formatting (old style)

**What it is:** the original, C-style formatting. Codes such as `%s`, `%d` and `%f` in the template are replaced by the values after the `%` operator. You'll still see it in older code and in the `logging` module.

| Code | Meaning |
|---|---|
| `%s` | a string (any value, converted with `str()`) |
| `%d` | an integer |
| `%f`, `%.2f` | a float, optionally with a number of decimals |
| `%5d`, `%-10s` | width 5, right-aligned; width 10, left-aligned |
| `%%` | a literal `%` sign |

In [10]:
print("%s scored %.2f" % (name, score))  # several values go in a tuple
print("%d items" % 7)                    # a single value doesn't need a tuple
print("[%5d]" % 42)                      # width 5, right-aligned
print("[%-10s]" % "left")                # width 10, left-aligned
print("Progress: %d%%" % 75)             # %% prints a literal %

Asha scored 92.46
7 items
[   42]
[left      ]
Progress: 75%


---
## 5. The format specification mini-language

Everything after the `:` inside `{}` is a **format spec**. It works the same way in `str.format()`, in f-strings, and in the built-in `format()` function.

```
{value:[fill][align][sign][width][,][.precision][type]}
```

| Part | Options | Example | Result |
|---|---|---|---|
| fill | any character | `"{:*>4}".format(7)` | `***7` |
| align | `<` left, `>` right, `^` centre | `"{:^6}".format("hi")` | `  hi  ` |
| sign | `+` always, `-` negatives only (default), space | `"{:+}".format(5)` | `+5` |
| width | minimum total width | `"{:6}".format(42)` | `    42` |
| grouping | `,` or `_` as a thousands separator | `"{:,}".format(1234567)` | `1,234,567` |
| precision | digits after the decimal point | `"{:.2f}".format(3.14159)` | `3.14` |
| type | `d` int, `f` fixed-point, `e` scientific, `%` percent, `b` binary, `o` octal, `x` hex, `s` string | `"{:x}".format(255)` | `ff` |

**Behind the scenes:** the built-in function `format(value, spec)` does the work in every style. Calling it directly gives the same result:

In [11]:
print(format(3.14159, ".2f"))     # the format() function
print("{:.2f}".format(3.14159))   # str.format()
print(f"{3.14159:.2f}")           # f-string

3.14
3.14
3.14


### 5.1 Numbers

In [12]:
pi = 3.14159265

print("{:.2f}".format(pi))              # 2 decimal places -> 3.14
print("{:.0f} {:.0f}".format(2.5, 3.5))  # 2 4: an exact half rounds to the even neighbour
print("{:,}".format(1234567))           # thousands separator -> 1,234,567
print("{:_}".format(1234567))           # underscore separator -> 1_234_567
print("{:,.2f}".format(1234567.891))    # separator and decimals together
print("{:.1%}".format(0.256))           # percent: multiplies by 100 -> 25.6%
print("{:e}".format(1234.5))            # scientific notation
print("{:+d} {:+d}".format(5, -5))      # always show the sign

3.14
2 4
1,234,567
1_234_567
1,234,567.89
25.6%
1.234500e+03
+5 -5


### 5.2 Number bases and zero padding

In [13]:
print("{:b} {:o} {:x} {:X}".format(255, 255, 255, 255))  # binary, octal, hex, HEX
print("{:#b} {:#x}".format(255, 255))                    # # adds the 0b / 0x prefix
print("{:05d}".format(42))                               # zero-pad to width 5 -> 00042
print("{:08.3f}".format(3.14159))                        # width 8 including the dot, 3 decimals

11111111 377 ff FF
0b11111111 0xff
00042
0003.142


### 5.3 Width and alignment

In [14]:
# Width 10; the brackets show where the padding goes
print("[{:<10}]".format("left"))
print("[{:>10}]".format("right"))
print("[{:^10}]".format("centre"))
print("[{:*^10}]".format("fill"))  # a custom fill character
print("[{:10}]".format(42))       # numbers are right-aligned by default
print("[{:10}]".format("text"))   # strings are left-aligned by default

[left      ]
[     right]
[  centre  ]
[***fill***]
[        42]
[text      ]


Alignment is what makes tidy text tables:

In [15]:
items = [("Tea", 3.5), ("Coffee", 12), ("Sandwich", 149.99)]

print("{:<10}|{:>8}".format("Item", "Price"))
print("-" * 19)
for item, price in items:
    print("{:<10}|{:>8.2f}".format(item, price))

Item      |   Price
-------------------
Tea       |    3.50
Coffee    |   12.00
Sandwich  |  149.99


---
## 6. Formatting dates

`datetime` objects accept date codes after the `:`, such as `%Y` for the year and `%d` for the day.

| Code | Meaning | Example |
|---|---|---|
| `%Y` | 4-digit year | `2026` |
| `%m` | month number | `10` |
| `%d` | day of the month | `17` |
| `%B` / `%b` | month name / short month name | `October` / `Oct` |
| `%A` / `%a` | weekday name / short weekday name | `Saturday` / `Sat` |
| `%H:%M` | hours and minutes (24-hour clock) | `18:30` |

In [16]:
from datetime import datetime

session = datetime(2026, 10, 17, 18, 30)  # a fixed date, so the output is the same on every run

print("{:%Y-%m-%d}".format(session))
print(f"{session:%d %B %Y}")
print(f"{session:%A, %d %b %Y at %H:%M}")

2026-10-17
17 October 2026
Saturday, 17 Oct 2026 at 18:30


---
## 7. Dynamic widths and literal braces

A placeholder can sit **inside** a format spec, so the width or precision can come from a variable. To print a real `{` or `}`, double it.

In [17]:
width, precision = 12, 3

# The width and precision are filled in from variables
print("[{:{}.{}f}]".format(pi, width, precision))
print(f"[{pi:{width}.{precision}f}]")

# {{ and }} print literal braces
print("{{literal braces}} and {}".format(name))
print(f"{{literal braces}} and {name}")

[       3.142]
[       3.142]
{literal braces} and Asha
{literal braces} and Asha


---
## 8. Common mistakes

| Mistake | What happens |
|---|---|
| Forgetting the `f` prefix | No error: the braces are printed as plain text |
| Fewer arguments than `{}` placeholders | `IndexError` |
| A named placeholder with no matching argument | `KeyError` |
| A number type on a string, like `{:d}` with `"abc"` | `ValueError` |
| Mixing `{}` and `{0}` in one template | `ValueError` |

In [18]:
# Forgetting the f prefix: no error, just the wrong output
print("{name} scored {score}")

{name} scored {score}


In [19]:
# Each line below raises an error, caught so the notebook keeps running
try:
    "{} and {}".format("only one")
except IndexError as error:
    print("IndexError:", error)

try:
    "{student} scored".format(name="Asha")
except KeyError as error:
    print("KeyError:", error)

try:
    "{:d}".format("abc")
except ValueError as error:
    print("ValueError:", error)

try:
    "{} and {0}".format("a", "b")
except ValueError as error:
    print("ValueError:", error)

IndexError: Replacement index 1 out of range for positional args tuple
KeyError: 'student'
ValueError: Unknown format code 'd' for object of type 'str'
ValueError: cannot switch from automatic field numbering to manual field specification


---
## 9. Which style should you use?

| Style | Use it when |
|---|---|
| f-string | Your default for new code: short and readable |
| `str.format()` | The template is stored separately from the values (in a variable, file or config), or reused many times |
| `%` | Reading older code, or in `logging` calls such as `logging.info("user %s logged in", name)` |

---
## 10. Practice exercises

Try each one yourself first, then compare with the solution.

**1. Shopping receipt.** Print each item left-aligned and its price right-aligned with 2 decimals, then a total with a thousands separator.

In [20]:
# Solution 1: receipt
cart = {"Laptop": 55999.0, "Mouse": 799.5, "Cable": 149.25}

line = "{:<12}{:>12,.2f}"  # one template, reused for every row
for item, price in cart.items():
    print(line.format(item, price))
print("-" * 24)
print(line.format("Total", sum(cart.values())))

Laptop         55,999.00
Mouse             799.50
Cable             149.25
------------------------
Total          56,947.75


**2. Report card.** For each student, print the name, marks out of 500, the percentage (1 decimal place) and Pass / Fail (40% or more is a pass), in aligned columns.

In [21]:
# Solution 2: report card
results = {"Asha": 456, "Ravi": 187, "Monal": 389}

header = "{:<8}{:>6}{:>8}{:>7}"
row = "{:<8}{:>6}{:>8.1%}{:>7}"  # the same widths, with a percent format for the third column

print(header.format("Name", "Marks", "Pct", "Result"))
print("-" * 29)
for student, marks in results.items():
    pct = marks / 500
    print(row.format(student, marks, pct, "Pass" if pct >= 0.4 else "Fail"))

Name     Marks     Pct Result
-----------------------------
Asha       456   91.2%   Pass
Ravi       187   37.4%   Fail
Monal      389   77.8%   Pass


**3. Mail merge.** Store one message template, then fill it in for each customer from a list of dicts. Show the amount with a thousands separator and 2 decimals, and the due date like `31 Oct 2026`.

In [22]:
# Solution 3: mail merge with a stored template
from datetime import date

reminder = "Hi {name}, your bill of Rs. {amount:,.2f} is due on {due:%d %b %Y}."

customers = [
    {"name": "Asha", "amount": 1250.5, "due": date(2026, 10, 31)},
    {"name": "Ravi", "amount": 98000, "due": date(2026, 11, 5)},
]

for customer in customers:
    print(reminder.format(**customer))

Hi Asha, your bill of Rs. 1,250.50 is due on 31 Oct 2026.
Hi Ravi, your bill of Rs. 98,000.00 is due on 05 Nov 2026.


**4. Progress bar.** Write a function that shows a 20-character text bar for a completion fraction, like `[########------------]  40%`.

In [23]:
# Solution 4: progress bar
def progress_bar(fraction, width=20):
    """Return a text progress bar for a fraction between 0 and 1."""
    filled = round(fraction * width)
    bar = "#" * filled + "-" * (width - filled)
    return "[{}] {:>4.0%}".format(bar, fraction)  # percent with 0 decimals, right-aligned in width 4

for fraction in (0, 0.4, 0.75, 1):
    print(progress_bar(fraction))

[--------------------]   0%
[########------------]  40%
[###############-----]  75%
[####################] 100%


---
## Summary

- `str.format()` fills `{}` placeholders in a template: positional `{}`, indexed `{0}`, or named `{name}`. It returns a new string.
- A `.format()` template can be stored and reused; `template.format(**data)` fills it from a dict.
- f-strings (`f"{expression}"`) are the shortest and most readable choice for new code. `{x=}` is handy for debugging.
- `%` formatting is the old style, still used in `logging`.
- The format spec after the `:` controls decimals (`.2f`), separators (`,`), percent (`%`), width and alignment (`<` `>` `^`), fill, sign and number base. It works the same in every style.
- `datetime` values take date codes such as `%d %b %Y`.
- Double the braces (`{{ }}`) to print them literally.